In [6]:
%pip install pandas             # Biblioteca principal para manipulação e análise de dados estruturados
%pip install openpyxl           # Dependência do Pandas para leitura de arquivos .xlsx
%pip install rich               # Estilização avançada e formatação de outputs no terminal/notebook

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
from rich import print
from rich.panel import Panel

# 1. Pipeline de Extração de Dados (Data Extraction)
O objetivo desta etapa é consolidar os múltiplos arquivos dispersos do Cadastro Nacional de Reclamações Fundamentadas (2009-2024) em uma estrutura unificada (DataFrame). 

Durante a extração, os seguintes tratamentos preliminares são aplicados:
* **Padronização de Cabeçalhos:** Conversão para minúsculas e mapeamento manual para o arquivo de 2017 (ausência de *header*).
* **Tratamento de Exceções (`on_bad_lines`):** Descarte de linhas com corrupção estrutural (vazamento de delimitadores) para garantir a integridade do *Data Warehouse*.
* **Rastreabilidade:** Injeção do metadado `ano_origem` para preservar a linhagem dos dados.

In [8]:
# Definindo o padrão de colunas (tudo em minúsculo) para forçar em todos os arquivos (principalmente em 2017 que está sem header)
colunas_padrao = [
    'anocalendario', 'dataarquivamento', 'dataabertura', 'codigoregiao', 'regiao',
    'uf', 'strrazaosocial', 'strnomefantasia', 'tipo', 'numerocnpj', 'radicalcnpj',
    'razaosocialrfb', 'nomefantasiarfb', 'cnaeprincipal', 'desccnaeprincipal',
    'atendida', 'codigoassunto', 'descricaoassunto', 'codigoproblema',
    'descricaoproblema', 'sexoconsumidor', 'faixaetariaconsumidor', 'cepconsumidor'
]


# Criando um dicionário com todos os caminhos dos arquivos, baseando-se no ano de cada arquivo
arquivos = {
    '2009': '../dados_brutos/CNRF_2009.csv',
    '2010': '../dados_brutos/CNRF_2010.csv',
    '2011': '../dados_brutos/CNRF_2011.csv',
    '2012': '../dados_brutos/CNRF_2012.csv',
    '2013': '../dados_brutos/CNRF_2013.csv',
    '2014': '../dados_brutos/CNRF_2014.csv',
    '2015': '../dados_brutos/CNRF_2015.csv',
    '2016': '../dados_brutos/CNRF_2016.csv',
    '2017': '../dados_brutos/CNRF_2017.csv',
    '2018': '../dados_brutos/CNRF_2018.csv',
    '2019': '../dados_brutos/CNRF_2019.csv',
    '2020': '../dados_brutos/CNRF_2020.csv',
    '2021': '../dados_brutos/CNRF_2021.csv',
    '2022': '../dados_brutos/CNRF_2022.csv',
    '2023': '../dados_brutos/CNRF_2023.csv',
    '2024': '../dados_brutos/CNRF_2024.xlsx'
}

lista_dataframes = []                       # Para armazenar os dataframes até podermos agregá-los em um só
total_linhas_brutas = 0                     # Contadores para aferirmos se estamos perdendo muitas linhas
total_linhas_limpas = 0

print("Iniciando a Extração de Dados dos arquivos CNRF...\n")

for ano, caminho_arquivo in arquivos.items():
    linhas_no_arquivo   = 0
    linhas_lidas_pandas = 0
    
    try:
        ### CONTAGEM BRUTA
        if caminho_arquivo.endswith('.csv'):
            # Lendo o arquivo como texto puro apenas para contar as linhas
            with open(caminho_arquivo, 'r', encoding='latin1') as f:
                linhas_no_arquivo = sum(1 for linha in f)
            
            # Subtraindo a primeira linha (header), exceto para 2017 que não tem 
            if ano != '2017':
                linhas_no_arquivo -= 1
                
        elif caminho_arquivo.endswith('.xlsx'):
            # No casos especial do arquivo em excel, vamos deixar o Pandas lidar sozinho com a contagem bruta
            df_temp = pd.read_excel(caminho_arquivo, dtype=str)
            linhas_no_arquivo = len(df_temp)

        ### LEITURA E LIMPEZA INICIAL COM PANDAS
        if caminho_arquivo.endswith('.csv'):
            if ano == '2017':
                # Caso especial em que o arquivo não apresenta header
                df = pd.read_csv(caminho_arquivo, encoding='latin1', sep=';', header=None, names=colunas_padrao, dtype=str, on_bad_lines='skip')
            else:
                # Caso comum em que o arquivo apresenta header
                df = pd.read_csv(caminho_arquivo, encoding='latin1', sep=';', dtype=str, on_bad_lines='skip')
                df.columns = colunas_padrao # Forçando o padrão
                
        elif caminho_arquivo.endswith('.xlsx'):
            df = df_temp # Aproveitando que o arquivo já foi lido
            df.columns = colunas_padrao # Forçando o padrão
        
        ### FINALIZAÇÃO DO ARQUIVO
        df['ano_origem'] = ano
        lista_dataframes.append(df)
        
        linhas_lidas_pandas = len(df)
        linhas_perdidas = linhas_no_arquivo - linhas_lidas_pandas
        
        # Atualização dos contadores globais
        total_linhas_brutas += linhas_no_arquivo
        total_linhas_limpas += linhas_lidas_pandas
        
        print(f"[bold cyan][{ano}][/] - Brutas: [blue]{linhas_no_arquivo:<7}[/] | Salvas: [green]{linhas_lidas_pandas:<7}[/] | Perdidas: [red]{linhas_perdidas}[/]")

    except Exception as e:
        print(f"[{ano}] - Erro Crítico: {e}")

### PRINT DOS RESULTADOS DA AUDITORIA
df_final = pd.concat(lista_dataframes, ignore_index=True)

taxa_perda = ((total_linhas_brutas - total_linhas_limpas) / total_linhas_brutas) * 100 if total_linhas_brutas > 0 else 0

print("-" * 40)
print("RESUMO DA EXTRAÇÃO")
print(f"Total de registros originais: {total_linhas_brutas}")
print(f"Total de registros recuperados: {total_linhas_limpas}")
print(f"Total de registros descartados: {total_linhas_brutas - total_linhas_limpas}")
print(f"Taxa de perda de dados: {taxa_perda:.4f}%")
print("-" * 40)

Iniciando a Extração de Dados dos arquivos CNRF...

[2009] - Brutas: 104869  | Salvas: 104869  | Perdidas: 0

[2010] - Brutas: 122664  | Salvas: 122664  | Perdidas: 0

[2011] - Brutas: 153096  | Salvas: 153096  | Perdidas: 0

[2012] - Brutas: 211076  | Salvas: 211076  | Perdidas: 0

...

Total de registros originais: 1802039

Total de registros recuperados: 1801980

Total de registros descartados: 59

Taxa de perda de dados: 0.0033%

----------------------------------------

# 2. Análise Exploratória e Diagnóstico de Dados (EDA)
Antes de iniciarmos os processos de transformação (Limpeza e Modelagem Dimensional), é crucial inspecionar a saúde estrutural da base consolidada, verificando os tipos inferidos, o consumo de memória e a integridade das instâncias.

In [9]:
# Verificando a estrutura do DataFrame consolidado
with pd.option_context('display.max_columns', None):
    display(df_final.head(5))
    
print("\n")
df_final.info()

print("\n--- Contagem de Valores Únicos por Coluna ---")
display(df_final.nunique(dropna=False))

,anocalendario,dataarquivamento,dataabertura,codigoregiao,regiao,uf,strrazaosocial,strnomefantasia,tipo,numerocnpj,radicalcnpj,razaosocialrfb,nomefantasiarfb,cnaeprincipal,desccnaeprincipal,atendida,codigoassunto,descricaoassunto,codigoproblema,descricaoproblema,sexoconsumidor,faixaetariaconsumidor,cepconsumidor,ano_origem
0,2009,2009-01-21 15:29:29.000,2005-07-06 08:32:23.000,05,Centro-oeste,GO,CBP SUL - COLCHÕES E ESPUMAS INDUSTRIAIS LTDA,LIMANSKY,1,01350934000116,01350934,CBP SUL - COLCHOES E ESPUMAS INDUSTRIAIS LTDA,NaN,3104700,FABRICAÇÃO DE COLCHÕES,S,100,Colchão,105,Produto entregue com danos/defeitos,M,entre 61 a 70 anos,74680330,2009
1,2009,2009-05-12 12:05:08.000,2006-01-17 12:10:53.000,03,Sudeste,RJ,GRADIENTE,NaN,1,43185362001936,43185362,IGB ELETRONICA S.A,NaN,2640000,"FABRICAÇÃO DE APARELHOS DE RECEPÇÃO, REPRODUÇÃ...",N,146,Aparelho DVD,107,Não entrega/demora na entrega do produto,F,entre 51 a 60 anos,23510240,2009
2,2009,2009-04-23 08:52:31.000,2005-07-06 11:28:47.000,05,Centro-oeste,GO,IGL INDÚSTRIA LTDA,ELIDA PONDS INDUSTRIAL LTDA,1,03085759000102,03085759,UNILEVER BRASIL HIGIENE PESSOAL E LIMPEZA LTDA,ELIDA PONDS INDUSTRIAL LTDA.,2063100,"FABRICAÇÃO DE COSMÉTICOS, PRODUTOS DE PERFUMAR...",N,224,Produto Para Uso Veterinário ( Medicamento / S...,177,Presença de sujidades/corpos estranhos,F,entre 21 a 30 anos,74775010,2009
3,2009,2009-04-23 08:52:31.000,2005-07-06 11:28:47.000,05,Centro-oeste,GO,UNILEVER BESTFOODS BRASIL LTDA,UNILEVER,1,01615814002066,01615814,UNILEVER BRASIL INDUSTRIAL LTDA,NaN,1031700,FABRICAÇÃO DE CONSERVAS DE FRUTAS,N,224,Produto Para Uso Veterinário ( Medicamento / S...,177,Presença de sujidades/corpos estranhos,F,entre 21 a 30 anos,74775010,2009
4,2009,2009-07-23 11:57:40.000,2006-01-18 15:35:29.000,03,Sudeste,RJ,EMPRESA BRASILEIRA DE TELECOMUNICAÇÕES S.A,EMBRATEL - LIVRE,1,33530486000129,33530486,EMPRESA BRASILEIRA DE TELECOMUNICACOES S A EMB...,NaN,6110899,SERVIÇOS DE TELECOMUNICAÇÕES POR FIO NÃO ESPEC...,S,186,Telefonia Fixa ( Plano de Expansão / Compra e ...,143,Contrato - Rescisão/alteração unilateral,F,entre 41 a 50 anos,20710270,2009


<class 'pandas.DataFrame'>
RangeIndex: 1801980 entries, 0 to 1801979
Data columns (total 24 columns):
 #   Column                 Dtype
---  ------                 -----
 0   anocalendario          str  
 1   dataarquivamento       str  
 2   dataabertura           str  
 3   codigoregiao           str  
 4   regiao                 str  
 5   uf                     str  
 6   strrazaosocial         str  
 7   strnomefantasia        str  
 8   tipo                   str  
 9   numerocnpj             str  
 10  radicalcnpj            str  
 11  razaosocialrfb         str  
 12  nomefantasiarfb        str  
 13  cnaeprincipal          str  
 14  desccnaeprincipal      str  
 15  atendida               str  
 16  codigoassunto          str  
 17  descricaoassunto       str  
 18  codigoproblema         str  
 19  descricaoproblema      str  
 20  sexoconsumidor         str  
 21  faixaetariaconsumidor  str  
 22  cepconsumidor          str  
 23  ano_origem             str  
dtypes: str(24

--- Contagem de Valores Únicos por Coluna ---

anocalendario                 21
dataarquivamento         1109174
dataabertura             1434018
codigoregiao                  11
regiao                         6
uf                            27
strrazaosocial            179075
strnomefantasia           124804
tipo                           3
numerocnpj                133214
radicalcnpj               103925
razaosocialrfb             91540
nomefantasiarfb            57350
cnaeprincipal               1029
desccnaeprincipal           1715
atendida                       3
codigoassunto                245
descricaoassunto             375
codigoproblema              3782
descricaoproblema            374
sexoconsumidor                 4
faixaetariaconsumidor         11
cepconsumidor             305552
ano_origem                    16
dtype: int64

### 2.1. Diagnóstico Detalhado por Coluna
Análise individual dos valores únicos das variáveis categóricas para mapear inconsistências de tipagem, erros de digitação e ruídos gerados na origem governamental.

In [10]:
# Analisando os valores únicos e a contagem de valores nulos para fundamentar o plano de limpeza
print(Panel("[bold bright_magenta]DIAGNÓSTICO DE VALORES ÚNICOS E NULOS POR COLUNA[/]", expand=False))

# Função auxiliar para padronizar o print e manter tudo alinhado
def print_diagnostico(nome_coluna):
    valores_unicos = df_final[nome_coluna].unique()
    qtd_nulos = df_final[nome_coluna].isnull().sum()
    print(f"[bold cyan]{nome_coluna:<22}[/]| Nulos: [bold red]{qtd_nulos:<6}[/]\n{valores_unicos}\n" + "-" * 80)

colunas_analise = [
    'anocalendario', 'codigoregiao', 'regiao', 'uf', 
    'tipo', 'atendida', 'sexoconsumidor', 'faixaetariaconsumidor', 'ano_origem'
]

for col in colunas_analise:
    print_diagnostico(col)

╭──────────────────────────────────────────────────╮
│ DIAGNÓSTICO DE VALORES ÚNICOS E NULOS POR COLUNA │
╰──────────────────────────────────────────────────╯

anocalendario         | Nulos: 3     
<StringArray>
[                    '2009',                        nan,
 '(104867 row(s) affected)',                     '2010',
 '(122662 row(s) affected)',                     '2011',
 '(153094 row(s) affected)',                     '2012',
                     '2013',                     '2014',
                     '2015',                     '2016',
                  'ï»¿2017',                     '2017',
                     '2018',                     '2019',
                     '2020',                     '2021',
                     '2022',                     '2023',
                     '2024']
Length: 21, dtype: str
--------------------------------------------------------------------------------

codigoregiao          | Nulos: 6     
<StringArray>
['05', '03', '01', '04', '02', nan, '2', '3', '4', '5', '1']
Length: 11, dtype: str
--------------------------------------------------------------------------------

regiao                | Nulos: 6     
<StringArray>
['Centro-oeste', 'Sudeste', 'Norte', 'Sul', 'Nordeste', nan]
Length: 6, dtype: str
--------------------------------------------------------------------------------

uf                    | Nulos: 6     
<StringArray>
['GO', 'RJ', 'PA', 'AP', 'SC', 'MT', 'AM', 'PB', 'BA', 'ES', 'MS', 'PE', 'PI',
 'RN', 'CE', 'MG', 'AC', 'AL', 'SE', 'DF', 'MA', 'TO',  nan, 'RS', 'SP', 'PR',
 'RO']
Length: 27, dtype: str
--------------------------------------------------------------------------------

tipo                  | Nulos: 6     
<StringArray>
['1', '0', nan]
Length: 3, dtype: str
--------------------------------------------------------------------------------

atendida              | Nulos: 6     
<StringArray>
['S', 'N', nan]
Length: 3, dtype: str
--------------------------------------------------------------------------------

sexoconsumidor        | Nulos: 1029  
<StringArray>
['M', 'F', 'N', nan]
Length: 4, dtype: str
--------------------------------------------------------------------------------

faixaetariaconsumidor | Nulos: 6     
<StringArray>
['entre 61 a 70 anos', 'entre 51 a 60 anos', 'entre 21 a 30 anos',
 'entre 41 a 50 anos', 'entre 31 a 40 anos',    'mais de 70 anos',
      'Nao Informada',        'até 20 anos',                  nan,
      'Nao se aplica',       'atÃ© 20 anos']
Length: 11, dtype: str
--------------------------------------------------------------------------------

ano_origem            | Nulos: 0     
<StringArray>
['2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
 '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Length: 16, dtype: str
--------------------------------------------------------------------------------

# 3. Plano de Ação para Transformação (Data Cleaning)
A análise exploratória das variáveis revelou inconsistências críticas nos registros brutos. A redundância de valores nulos simultâneos (6 registros) indica a presença de linhas fantasmas derivadas de logs de banco de dados da origem (ex: `(104867 row(s) affected)`). 

Para garantir a qualidade da base antes da modelagem em Esquema Estrela, as seguintes transformações serão aplicadas nesta ordem:

1.  **Limpeza Global de Ruídos:** Remoção das instâncias vazias/corrompidas geradas por metadados da exportação original, que afetam simultaneamente as colunas `regiao`, `uf`, `tipo`, `atendida` e `faixaetariaconsumidor`.
<br><br>

2.  **`anocalendario`:** A coluna será descartada (Drop). A coluna `ano_origem`, gerada no pipeline de extração, encontra-se totalmente íntegra e assumirá a função de eixo temporal primário.
<br><br>

3.  **`codigoregiao`:** Padronização da máscara numérica. Entradas do tipo `'0X'` serão tratadas e convertidas para o padrão inteiro `'X'`.
<br><br>

4.  **`sexoconsumidor`:** Padronização categórica. Entradas irregulares (`'N'`) e valores nulos (`NaN`) serão unificados sob a categoria `'OUTROS'`.
<br><br>

5.  **`faixaetariaconsumidor`:** A coluna passará por uma higienização textual profunda em uma etapa isolada devido à alta variabilidade tipográfica e de formatação.

In [13]:
print(Panel("[bold bright_magenta]Iniciando a Limpeza Inicial dos Dados...[/]", expand=False))

# Tamanho original para calcularmos as métricas de perda
quantidade_linhas_antes = len(df_final) 



### Descarte do 'anocalendario'
df_final = df_final.drop(columns=['anocalendario'], errors='ignore')


### Limpeza Global de Ruídos (NaN)
df_final.dropna(how='all', inplace=True) # Linhas completamente vazias
df_final = df_final.dropna(subset=['codigoregiao', 'regiao', 'uf', 'tipo', 'atendida', 'faixaetariaconsumidor'])


### Padronização do 'codigoregiao' | 0X -> X
padrao_codigo_regiao = {
    "01" : "1", 
    "02" : "2", 
    "03" : "3", 
    "04" : "4", 
    "05" : "5"
}
df_final['codigoregiao'] = df_final['codigoregiao'].replace(padrao_codigo_regiao) # Faz a substituição conforme o dicionário 'padrao_codigo_regiao'


### Padronização do 'sexoconsumidor' | N, NaN -> OUTROS
df_final['sexoconsumidor'] = df_final['sexoconsumidor'].fillna("OUTROS") # Preenche valores nulos (NaN)
df_final['sexoconsumidor'] = df_final['sexoconsumidor'].replace({"N" : "OUTROS"})


### PRINT DOS RESULTADOS DA AUDITORIA
linhas_descartadas = quantidade_linhas_antes - len(df_final)
porcentagem_perda = (linhas_descartadas / quantidade_linhas_antes) * 100 if quantidade_linhas_antes > 0 else 0

print(f"[bold green]Limpeza concluída com sucesso![/]")
print(f"Total de linhas antes:  [cyan]{quantidade_linhas_antes}[/]")
print(f"Total de linhas válidas:[bold cyan] {len(df_final)}[/]")
print(f"Linhas descartadas:     [red]{linhas_descartadas}[/] ({porcentagem_perda:.4f}%)")
print("-" * 60)


for col in colunas_analise:
    if col != 'anocalendario': # Evitando erro porque descartamos essa coluna
        print_diagnostico(col)

╭──────────────────────────────────────────╮
│ Iniciando a Limpeza Inicial dos Dados... │
╰──────────────────────────────────────────╯

Limpeza concluída com sucesso!

Total de linhas antes:  1801974

Total de linhas válidas: 1801974

Linhas descartadas:     0 (0.0000%)

...

tipo                  | Nulos: 0     
<StringArray>
['1', '0']
Length: 2, dtype: str
--------------------------------------------------------------------------------

atendida              | Nulos: 0     
<StringArray>
['S', 'N']
Length: 2, dtype: str
--------------------------------------------------------------------------------

sexoconsumidor        | Nulos: 0     
<StringArray>
['M', 'F', 'OUTROS']
Length: 3, dtype: str
--------------------------------------------------------------------------------

faixaetariaconsumidor | Nulos: 0     
<StringArray>
['entre 61 a 70 anos', 'entre 51 a 60 anos', 'entre 21 a 30 anos',
 'entre 41 a 50 anos', 'entre 31 a 40 anos',    'mais de 70 anos',
      'Nao Informada',        'até 20 anos',      'Nao se aplica',
       'atÃ© 20 anos']
Length: 10, dtype: str
--------------------------------------------------------------------------------

ano_origem            | Nulos: 0     
<StringArray>
['2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017',
 '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Length: 16, dtype: str
--------------------------------------------------------------------------------